# CardioWave AI: Subgroup Audit

Explore feature-defined subgroups; this is not a protected-group fairness guarantee.

**Synthetic educational data only.** Each notebook is independently runnable. Validation analyses do not establish clinical effectiveness.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Run from project root or notebooks directory')
sys.path.insert(0,str(ROOT/'src'))
from healthcare_ai.settings import FEATURES,TARGET
from healthcare_ai.io import load_csv
from healthcare_ai.splits import split
from healthcare_ai.pipeline import build
from healthcare_ai.metrics import evaluate
frame=load_csv(ROOT/'data/demo.csv')
train,val,test=split(frame)
model=build().fit(train[FEATURES],train[TARGET])


In [2]:
from healthcare_ai.fairness import subgroup_metrics
p=model.predict_proba(val[FEATURES])[:,1]
groups=np.where(val[FEATURES[0]]>=train[FEATURES[0]].median(),'above_training_median','below_training_median')
print(json.dumps(subgroup_metrics(val[TARGET],p,groups),indent=2))

{
  "above_training_median": {
    "auroc": 0.8799999999999999,
    "average_precision": 0.9239335298060685,
    "brier": 0.13425169151937658,
    "balanced_accuracy": 0.75,
    "sensitivity": 0.8,
    "specificity": 0.7,
    "tp": 32,
    "tn": 14,
    "fp": 6,
    "fn": 8,
    "n": 60,
    "threshold": 0.5
  },
  "below_training_median": {
    "auroc": 0.8181818181818182,
    "average_precision": 0.8291337194962254,
    "brier": 0.1722955025719113,
    "balanced_accuracy": 0.7457912457912458,
    "sensitivity": 0.7037037037037037,
    "specificity": 0.7878787878787878,
    "tp": 19,
    "tn": 26,
    "fp": 7,
    "fn": 8,
    "n": 60,
    "threshold": 0.5
  }
}


## Interpretation

These outputs describe a mathematical simulator. Review sample size, uncertainty and the documented data contract before adapting this analysis. Repeated validation inspection can overfit development choices; use a separate frozen holdout for final assessment.